<a href="https://colab.research.google.com/github/KT313-cloud/study/blob/main/03_linear_regression_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python 演習

本ページのコードの多くは、An Introduction to Statistical Learning の演習のコードを Python 用に書き換えたものです。以下の文献やサイトを参考に作成されています。

 - https://www.statlearning.com/
 - https://github.com/qx0731/ISL_python
 - https://matplotlib.org/stable/tutorials/introductory/usage.html


## 線形回帰


### モジュールの読み込み

- `statsmodels`
  - 統計モデルの推定のためのクラスや関数を提供するモジュール
- `sklearn`
  - scikit-learn: Python 用の機械学習ライブラリ。statsmodels が統計分析を主な目的とするのに対し、scikit-learn は機械学習（出力値の予測）が主目的。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.graphics.regressionplots import *
from sklearn import datasets, linear_model


### California Housing データセットの読み込み


In [ ]:
from sklearn.datasets import fetch_california_housing
california_data = fetch_california_housing()  # California Housing データセットを読み込み
print(california_data.DESCR)                  # データの説明を表示
California = pd.DataFrame(california_data.data, columns=california_data.feature_names)  # データフレームに変換し、列名を設定
print(list(California))   # 列名を表示
print(California.shape)   # データのサイズを表示
California                # データを表示

予測対象のデータ（Median House Value）を追加

In [ ]:
California['MedHouseValue'] = california_data.target  # Median House Value の列を追加
#California.drop(California[California['MedHouseValue'] > 5].index, inplace=True)  # Median House Value が clip されている行を削除
California

## 線形単回帰

- `ols()`
  - OLS (Ordinary Least Squares)
  - `ols(y ~ x, data)`で `x` を説明変数、 `y` を応答変数として線形モデル（linear model）を最小二乗法でフィッテイング（パラメータ推定）


In [ ]:
lm = smf.ols('MedHouseValue ~ MedInc', data=California).fit()   # 線形回帰モデルを学習
lm.summary()

### フィットさせたモデルの情報

パラメータの値

In [ ]:
lm.params

信頼区間

In [ ]:
lm.conf_int()

### フィットさせたモデルを用いて予測

- `predict()`

In [ ]:
lm.predict(pd.DataFrame({'MedInc':[5, 10, 15]}))  # MedInc = 5, 10, 15 を含むデータフレームを作成して予測値を計算

### プロット



In [ ]:
California.plot(kind='scatter', x='MedInc', y='MedHouseValue')
plt.show()
range = pd.DataFrame({'MedInc': [California.MedInc.min(), California.MedInc.max()]})  # 入力変数の最小値と最大値を取得
preds = lm.predict(range)                                                             # 最小値と最大値に対して予測値を計算
California.plot(kind='scatter', x='MedInc', y='MedHouseValue')
plt.plot(range, preds, c='red', linewidth=2)                                          # 予測値を結んで回帰直線をプロット
plt.show()


### 残差プロット

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2)  # 2x2 のサブプロットを作成
ax1.plot(California.MedInc, lm.predict(),'ro')      # 実際の Median Income と予測された Median House Value
ax2.plot(lm.predict(), lm.resid, 'go')              # 予測された Median House Value と残差
ax3.plot(lm.predict(), lm.resid_pearson, 'bo')      # 予測された Median House Value と標準化された残差 (分散 = 1)
plt.show()

### scikit-learn による線形回帰


In [ ]:
x = pd.DataFrame(California.MedInc)       # 'MedInc'列をデータフレームに変換し、入力変数として使用
y = California.MedHouseValue              # 'MedHouseValue'列を出力変数として使用
print(x.shape)
model = linear_model.LinearRegression()   # 線形回帰モデルのインスタンスを作成
model.fit(x, y)                           # 線形回帰モデルの学習
print(model.intercept_)                   # β_0を表示
print(model.coef_)                        # β_1を表示